# imports

In [ ]:
from loguru import logger
from tqdm import tqdm

# add-ons

## Public datasets you can ingest immediately:

Hugging Face: S&P500 earnings transcripts 2005–2025 (dataset card). 
Hugging Face

Strux: ~11,950 transcripts (2017–2024), nicely packaged for NLP. 
struxdata.github.io

Free web sources (great for fresh calls, but check the site ToS before crawling):

Motley Fool hosts many transcripts openly. 
The Motley Fool
+2
The Motley Fool
+2

Seeking Alpha also posts transcripts (often behind a soft paywall). 
Seeking Alpha
+2
Seeking Alpha
+2

Commercial APIs with clean licensing: Quartr and others. 
quartr.dev
+2
S&P Global Marketplace
+2

Practical approach: start with SEC 10-Ks (clean licensing), then add 100–500 transcripts from an open dataset for variety. Keep a source field in metadata so your UI can cite precisely.

## Making the text RAG-friendly

10-Ks are long. Good chunking = better answers. Try section-aware splits using the canonical headings:

Item 1\. Business

Item 1A\. Risk Factors

Item 7\. Management’s Discussion and Analysis

Item 7A\. Quantitative and Qualitative Disclosures about Market Risk

Regex the headings in the .txt and split by sections before your usual token-level chunker. (It dramatically improves retrieval precision on “MD&A” and “Risk Factors”.)

# initial script to fetch 10Ks over a list of stock tickers

What it does

Map tickers → CIKs using SEC JSON.

For each CIK, hit the Submissions API, filter last N 10-Ks, construct the filing URLs, and download the primary HTML.

Save text + metadata (CIK, ticker, year, accession number) for RAG.

This follows SEC guidance (User-Agent, polite rate). The Submissions API is updated in near real-time as filings are disseminated.

Bulk zips (nightly)
- submissions.zip and companyfacts.zip if you need to mirror lots of data quickly.

In [11]:
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from loguru import logger
from lxml.html import fromstring
from readability import Document
from tqdm import tqdm

# required by SEC policies
UA = "MinHtoo linmin.htoo@gmail.com"
session = requests.Session()
session.headers.update({"User-Agent": UA, "Accept-Encoding": "gzip, deflate"})



# TODO: define proper return dataclass
def ticker_map(timeout: int = 30):
    # Official SEC static mapping of tickers <-> CIKs
    j = session.get("https://www.sec.gov/files/company_tickers.json", timeout=timeout).json()
    # normalize -> { "AAPL": "0000320193", ... }
    m = {}
    for _, row in j.items():
        cik = f"{int(row['cik_str']):010d}"
        m[row["ticker"].upper()] = cik
    return m


def list_10k_submissions(cik, max_docs: int = 2, timeout: int = 30):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    s = session.get(url, timeout=timeout).json()
    filings = s.get("filings", {}).get("recent", {})
    forms, acc, prim, dates = [filings.get(k, []) for k in ("form", "accessionNumber", "primaryDocument", "filingDate")]
    out = []
    for f, a, p, d in zip(forms, acc, prim, dates):
        # we only want 10Ks, ignore the rest
        if f == "10-K":
            out.append((a.replace("-", ""), p, d))
        if len(out) >= max_docs:
            break
    return out  # [(accession_no_nohyphens, primary_doc, filing_date), ...]


def download_primary(cik, acc_no, primary_doc, timeout: int = 60):
    base = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no}/"
    url = urljoin(base, primary_doc)
    html = session.get(url, timeout=timeout).text
    return html, url


def strip_html_to_text(html):
    # Remove script/style sections
    # FIXED: </\1> instead of </\\1>
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", html)
    # Replace <br> with newline
    text = re.sub(r"(?is)<br\s*/?>", "\n", text)
    # Replace </p> with double newline
    text = re.sub(r"(?is)</p>", "\n\n", text)
    # Remove all other HTML tags
    text = re.sub(r"(?is)<.*?>", " ", text)
    # Collapse spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse excessive newlines
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def extract_text_readability(html: str) -> str:
    doc = Document(html)
    summary = doc.summary()
    tree = fromstring(summary)
    return tree.text_content().strip()


def fetch_10ks_for_tickers(
    tickers: list[str],
    output_dir: Path,
    per_company: int = 2,
    delay: float = 0.2
):
    out_raw_folder = output_dir / "10k_raw"
    out_meta_folder = output_dir / "meta"
    out_raw_folder.mkdir(parents=True, exist_ok=True)
    out_meta_folder.mkdir(parents=True, exist_ok=True)

    t2c = ticker_map()
    for t in tqdm(tickers, desc="fetching tickers"):
        cik = t2c[t.upper()]
        for acc_no, primary, fdate in tqdm(list_10k_submissions(cik, per_company), desc=f"processing ticker {t}"):
            html, src = download_primary(cik, acc_no, primary)

            # text = strip_html_to_text(html)
            text = extract_text_readability(html)
            base = f"{t.upper()}_{acc_no}"

            (out_raw_folder / f"{base}.html").write_text(html, encoding="utf-8")
            (out_raw_folder / f"{base}.txt").write_text(text, encoding="utf-8")

            meta = {
                "ticker": t.upper(),
                "cik": cik,
                "filing_date": fdate,
                "accession": acc_no,
                "primary": primary,
                "source_url": src,
                "form": "10-K",
            }
            (out_meta_folder / f"{base}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

            time.sleep(delay)  # be polite

    logger.success("Done.")


# if __name__ == "__main__":
#     # TODO: get list of tickers from somewhere, e.g. top 500 companies
#     tickers = [
#         "AAPL",
#         "MSFT",
#         "AMZN"
#     ]
#     fetch_10ks_for_tickers(tickers, per_company=5, delay=0.2)


In [13]:
tickers = [
    "APH",
    "GOOGL",
    "NVDA",
]
out_dir = Path("test")
fetch_10ks_for_tickers(tickers, out_dir, per_company=5, delay=0.2)

fetching tickers: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:12<00:00,  4.21s/it]
2025-10-19 01:59:16.303 | SUCCESS  | __main__:fetch_10ks_for_tickers:114 - Done.


# APH -> readability failed to convert HTML into text

## trying GPT5 suggestion using lxml -> still doesnt work, only a small paragraph again

In [14]:
import re
from lxml.html import fromstring, tostring
from readability import Document

def _normalize_ws(text: str) -> str:
    # collapse spaces/tabs, normalize newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\u00A0", " ", text)  # &nbsp;
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_text_lxml_preserve_tables(html: str) -> str:
    tree = fromstring(html)

    # Drop obvious non-content
    for bad in tree.xpath("//script|//style|//noscript|//iframe|//svg|//form|//nav|//header|//footer"):
        bad.getparent().remove(bad)

    # Insert structural separators so text_content() doesn't mush everything together
    # Paragraphs / breaks
    for el in tree.xpath("//br"):
        el.drop_tag()  # keep tail
        if el.tail is None:
            el.tail = "\n"
        else:
            el.tail = el.tail + "\n"

    for el in tree.xpath("//p|//div"):
        # add blank line after paragraphs / blocks
        el.tail = (el.tail or "") + "\n\n"

    # Lists
    for el in tree.xpath("//li"):
        el.text = (el.text or "")
        # bullet-ish prefix helps readability in plain text
        el.text = ("- " + el.text) if not el.text.startswith("- ") else el.text
        el.tail = (el.tail or "") + "\n"

    # Tables: separate cells with tabs and rows with newlines
    for el in tree.xpath("//td|//th"):
        el.tail = (el.tail or "") + "\t"
    for el in tree.xpath("//tr"):
        el.tail = (el.tail or "") + "\n"

    # Add line breaks after headings to preserve SEC section markers
    for el in tree.xpath("//h1|//h2|//h3|//h4|//h5|//h6"):
        el.tail = (el.tail or "") + "\n\n"

    text = tree.text_content()
    return _normalize_ws(text)

def extract_text_readability_first(html: str, min_chars: int = 5000) -> str:
    """
    Try readability; if it returns too little (common for table-heavy filings or index pages),
    fall back to an lxml extractor that preserves tables and structure.
    """
    try:
        doc = Document(html)
        # doc.summary() returns an HTML fragment; parse and get text
        summary_html = doc.summary()
        tree = fromstring(summary_html)
        text = _normalize_ws(tree.text_content())
    except Exception:
        text = ""

    if len(text) >= min_chars:
        return text

    # Fallback: robust structural extraction (handles tables, IXBRL-heavy docs)
    return extract_text_lxml_preserve_tables(html)


## try lxml -> ok, pretty good. just need to remove a lot of empty lines. better than regex as regex fails to parse certain symbols and characters, eg quotes, and bullet points.

In [24]:
html_content = Path("./test/10k_raw/APH_000155837025000714.html").read_bytes()
text = extract_text_lxml_preserve_tables(html_content)
Path("APH_debug_lxml.txt").write_text(text, encoding="utf-8")
print(len(text))

525047


## fallback to old regex parsing (fixed bug of removing random chars) - yes it works, BUT a lot of random symbols in the output too. need to clean up...

In [ ]:

def strip_html_to_text(html):
    # Remove script/style sections
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", html)
    # Replace <br> with newline
    text = re.sub(r"(?is)<br\s*/?>", "\n", text)
    # Replace </p> with double newline
    text = re.sub(r"(?is)</p>", "\n\n", text)
    # Remove all other HTML tags
    text = re.sub(r"(?is)<.*?>", " ", text)
    # Collapse spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse excessive newlines
    return re.sub(r"\n{3,}", "\n\n", text).strip()

In [16]:
html_content = Path("./test/10k_raw/APH_000155837025000714.html").read_text()
text = strip_html_to_text(html_content)
Path("APH_debug_regex.txt").write_text(text, encoding="utf-8")
print(len(text))

608011


# try latest version with lxml -> clean blank lines

In [27]:
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from loguru import logger
from lxml.html import fromstring
from tqdm import tqdm

# required by SEC policies
UA = "MinHtoo linmin.htoo@gmail.com"
session = requests.Session()
session.headers.update({"User-Agent": UA, "Accept-Encoding": "gzip, deflate"})

# Characters that often sneak in from SEC HTML
# - Zero-width: U+200B..U+200D, U+2060, U+FEFF
# - Soft hyphen: U+00AD
# - Non-breaking / thin / en space variants: U+00A0, U+202F, U+2000..U+200A, U+205F, U+3000
_INVISIBLES_RE = re.compile(r"[\u200B-\u200D\u2060\uFEFF\u00AD]")
_UNISPACES_RE = re.compile(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]")


# TODO: define proper return dataclass
def ticker_map(timeout: int = 30):
    # Official SEC static mapping of tickers <-> CIKs
    j = session.get("https://www.sec.gov/files/company_tickers.json", timeout=timeout).json()
    # normalize -> { "AAPL": "0000320193", ... }
    m = {}
    for _, row in j.items():
        cik = f"{int(row['cik_str']):010d}"
        m[row["ticker"].upper()] = cik
    return m


def list_10k_submissions(cik, max_docs: int = 2, timeout: int = 30):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    s = session.get(url, timeout=timeout).json()
    filings = s.get("filings", {}).get("recent", {})
    forms, acc, prim, dates = [filings.get(k, []) for k in ("form", "accessionNumber", "primaryDocument", "filingDate")]
    out = []
    for f, a, p, d in zip(forms, acc, prim, dates):
        # we only want 10Ks, ignore the rest
        if f == "10-K":
            out.append((a.replace("-", ""), p, d))
        if len(out) >= max_docs:
            break
    return out  # [(accession_no_nohyphens, primary_doc, filing_date), ...]


def download_primary(cik, acc_no, primary_doc, timeout: int = 60):
    base = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no}/"
    url = urljoin(base, primary_doc)
    html = session.get(url, timeout=timeout).text
    return html, url


def strip_html_to_text(html):
    # Remove script/style sections
    # FIXED: </\1> instead of </\\1>
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", html)
    # Replace <br> with newline
    text = re.sub(r"(?is)<br\s*/?>", "\n", text)
    # Replace </p> with double newline
    text = re.sub(r"(?is)</p>", "\n\n", text)
    # Remove all other HTML tags
    text = re.sub(r"(?is)<.*?>", " ", text)
    # Collapse spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse excessive newlines
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def _normalize_ws(text: str) -> str:
    # collapse spaces/tabs, normalize newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\u00A0", " ", text)  # &nbsp;
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_lxml_preserve_tables(html: str) -> str:
    tree = fromstring(html.encode("utf-8"))

    # Drop obvious non-content
    for bad in tree.xpath("//script|//style|//noscript|//iframe|//svg|//form|//nav|//header|//footer"):
        bad.getparent().remove(bad)

    # Insert structural separators so text_content() doesn't mush everything together
    # Paragraphs / breaks
    for el in tree.xpath("//br"):
        el.drop_tag()  # keep tail
        if el.tail is None:
            el.tail = "\n"
        else:
            el.tail = el.tail + "\n"

    for el in tree.xpath("//p|//div"):
        # add blank line after paragraphs / blocks
        el.tail = (el.tail or "") + "\n\n"

    # Lists
    for el in tree.xpath("//li"):
        el.text = el.text or ""
        # bullet-ish prefix helps readability in plain text
        el.text = ("- " + el.text) if not el.text.startswith("- ") else el.text
        el.tail = (el.tail or "") + "\n"

    # Tables: separate cells with tabs and rows with newlines
    for el in tree.xpath("//td|//th"):
        el.tail = (el.tail or "") + "\t"
    for el in tree.xpath("//tr"):
        el.tail = (el.tail or "") + "\n"

    # Add line breaks after headings to preserve SEC section markers
    for el in tree.xpath("//h1|//h2|//h3|//h4|//h5|//h6"):
        el.tail = (el.tail or "") + "\n\n"

    text = tree.text_content()
    return _normalize_ws(text)


def clean_invisible_and_blank_lines(text: str, keep_blank: int = 1) -> str:
    # 1) Remove zero-width & formatting chars entirely
    text = _INVISIBLES_RE.sub("", text)

    # 2) Normalize odd Unicode spaces to a plain space
    text = _UNISPACES_RE.sub(" ", text)

    # 3) Trim trailing spaces on each line
    text = re.sub(r"[ \t]+$", "", text, flags=re.M)

    # 4) Collapse runs of lines that are empty (after normalization)
    # keep_blank=1 -> at most one empty line between blocks
    # Convert multiple blank lines to exactly keep_blank
    if keep_blank >= 0:
        # First, normalize any lines containing only whitespace to just "\n"
        text = re.sub(r"(?m)^[ \t]*\n", "\n", text)
        # Then collapse runs of newlines
        limit = keep_blank + 1  # e.g., keep 1 blank -> at most two consecutive "\n"
        text = re.sub(r"\n{%d,}" % (limit + 1), "\n" * limit, text)

    return text.strip()


def fetch_10ks_for_tickers(tickers: list[str], output_dir: Path, per_company: int = 2, delay: float = 0.2):
    out_raw_folder = output_dir / "10k_raw"
    out_meta_folder = output_dir / "meta"
    out_raw_folder.mkdir(parents=True, exist_ok=True)
    out_meta_folder.mkdir(parents=True, exist_ok=True)

    t2c = ticker_map()
    for t in tqdm(tickers, desc="fetching tickers"):
        cik = t2c[t.upper()]
        for acc_no, primary, fdate in tqdm(list_10k_submissions(cik, per_company), desc="processing ticker"):
            html, src = download_primary(cik, acc_no, primary)

            # text = strip_html_to_text(html)
            text = extract_text_lxml_preserve_tables(html)
            # keep at most one blank line
            text = clean_invisible_and_blank_lines(text, keep_blank=1)

            base = f"{t.upper()}_{acc_no}"

            (out_raw_folder / f"{base}.html").write_text(html, encoding="utf-8")
            (out_raw_folder / f"{base}.txt").write_text(text, encoding="utf-8")

            meta = {
                "ticker": t.upper(),
                "cik": cik,
                "filing_date": fdate,
                "accession": acc_no,
                "primary": primary,
                "source_url": src,
                "form": "10-K",
            }
            (out_meta_folder / f"{base}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

            time.sleep(delay)  # be polite

    logger.success("Done.")

In [28]:
tickers = [
    "APH",
    "GOOGL",
    "NVDA",
]
out_dir = Path("test_v2")
fetch_10ks_for_tickers(tickers, out_dir, per_company=5, delay=0.2)

fetching tickers: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.30s/it]
2025-10-19 03:45:36.448 | SUCCESS  | __main__:fetch_10ks_for_tickers:182 - Done.


# try to parse tables into Markdown

In [ ]:
import re
from io import StringIO
from typing import List, Tuple, Dict, Any

from lxml.html import fromstring, tostring
import pandas as pd

# --- helpers for cleanup ---
_INVISIBLES_RE = re.compile(r"[\u200B-\u200D\u2060\uFEFF\u00AD]")
_UNISPACES_RE  = re.compile(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]")

def _clean_cell(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x)
    s = _INVISIBLES_RE.sub("", s)
    s = _UNISPACES_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # normalize standalone dashes/empties often used for zeros/NA
    return s

def _flatten_multiindex_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" | ".join([_clean_cell(c) for c in tup if _clean_cell(c)])
                      for tup in df.columns.values]
    else:
        df.columns = [_clean_cell(c) for c in df.columns]
    return df

def _drop_empty_rows_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(how="all").copy()
    df.columns = [c if c else f"col_{i}" for i, c in enumerate(df.columns)]
    # Drop columns that are entirely empty
    empty_cols = [c for c in df.columns if df[c].astype(str).str.strip().replace("nan","").eq("").all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)
    # Drop rows that are entirely empty after cleaning
    df = df[~df.apply(lambda r: all((_clean_cell(v) == "") for v in r), axis=1)]
    return df

def _table_caption(el, max_length: int = 1000) -> str:
    # prefer <caption>, fall back to previous sibling heading, else empty
    cap = ""
    caps = el.xpath(".//caption")
    if caps and caps[0].text_content().strip():
        cap = caps[0].text_content().strip()
    else:
        # Walk up to find a preceding heading
        prev = el.getprevious()
        while prev is not None and not re.match(r"^h[1-6]$", prev.tag or "", re.I):
            prev = prev.getprevious()
        if prev is not None:
            cap = prev.text_content().strip()
    # clean up caption
    cap = _clean_cell(cap)
    # Avoid super long captions
    return cap[:max_length]

def extract_tables_markdown_and_facts(html_bytes: bytes) -> List[Dict[str, Any]]:
    """
    Returns a list of dicts:
    {
      'caption': str,
      'markdown': str,   # GitHub-style table
      'facts': [str],    # row-wise sentences
      'n_rows': int,
      'n_cols': int
    }
    """
    tree = fromstring(html_bytes)
    tables = tree.xpath("//table")
    out = []

    for t in tables:
        frag_html = tostring(t, encoding="unicode")
        try:
            dfs = pd.read_html(StringIO(frag_html), flavor="lxml")
        except Exception:
            # fallback: try html5lib if available
            try:
                dfs = pd.read_html(StringIO(frag_html), flavor="bs4")
            except Exception:
                continue  # skip malformed table

        if not dfs:
            continue

        # Some SEC tables parse into multiple frames; concatenate vertically when shapes align
        df = pd.concat(dfs, ignore_index=True, sort=False)

        df = _flatten_multiindex_columns(df)
        # Clean cells
        # df = df.applymap(_clean_cell)
        df = df.map(_clean_cell)
        # Drop empties
        df = _drop_empty_rows_cols(df)

        if df.empty or df.shape[1] < 2:
            continue

        # Identify row label column (heuristic: first non-numeric-heavy column)
        row_label_col = df.columns[0]
        # Produce Markdown
        md = df.to_markdown(index=False)

        # Build facts: “<caption>: <row> — <col> = <value>”
        caption = _table_caption(t)
        facts = []
        for _, row in df.iterrows():
            row_label = row.get(row_label_col, "")
            if not row_label:
                continue
            for col in df.columns:
                if col == row_label_col:
                    continue
                val = row.get(col, "")
                if val == "":
                    continue
                fact = f"{caption}: {row_label} — {col} = {val}" if caption else f"{row_label} — {col} = {val}"
                facts.append(fact)

        out.append({
            "caption": caption,
            "markdown": md,
            "facts": facts,
            "n_rows": int(df.shape[0]),
            "n_cols": int(df.shape[1]),
        })
    return out


In [54]:
html_bytes = Path("./test_v2/10k_raw/APH_000155837025000714.html").read_bytes()
tables = extract_tables_markdown_and_facts(html_bytes)

In [55]:
len(tables)

120

### cols are split up?
- e.g. `tables[100]` -> `$ | 17.8 | $ | 19.5` , but it is already pretty impressive I must say
- `tables[70]` -> very nice

follow ups
- how to parse captions? eg `tables[70]` -> caption is empty
- example caption in HTML (it does not have "caption" tag, just a string, i guess it's difficult to parse this with rules)
     - </p><p style="font-family:'Times New Roman','Times','serif';font-size:10pt;text-indent:18pt;margin:0pt;">The following table reconciles Adjusted Operating Income, Adjusted Operating Margin, Adjusted Net Income attributable to Amphenol Corporation, Adjusted Effective Tax Rate and Adjusted Diluted EPS (each as defined in the &#8220;Non-GAAP Financial Measures&#8221; section below) to the most directly comparable U.S. GAAP financial measures for the years ended December 31, 2024 and 2023:</p>

In [56]:
tables[70]

{'caption': '',
 'markdown': '| 0                                                                   | 2                       | 3                       | 4                       | 5                       | 6                       | 7                       | 8                       | 9                       |\n|:--------------------------------------------------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|\n|                                                                     | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, |\n|                                                                     | 2024                    | 2024                    |    

In [57]:
print(tables[70]["markdown"])

| 0                                                                   | 2                       | 3                       | 4                       | 5                       | 6                       | 7                       | 8                       | 9                       |
|:--------------------------------------------------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|:------------------------|
|                                                                     | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, | Year Ended December 31, |
|                                                                     | 2024                    | 2024                    |                         | 2023      

In [58]:
for i, table in enumerate(tables):
    print(f"{i=}, {table['caption']}")

i=0, 
i=1, 
i=2, 
i=3, 
i=4, 
i=5, 
i=6, 
i=7, 
i=8, 
i=9, 
i=10, 
i=11, 
i=12, 
i=13, 
i=14, 
i=15, 
i=16, 
i=17, 
i=18, 
i=19, 
i=20, 
i=21, 
i=22, 
i=23, 
i=24, 
i=25, 
i=26, 
i=27, 
i=28, 
i=29, 
i=30, 
i=31, 
i=32, 
i=33, 
i=34, 
i=35, 
i=36, 
i=37, 
i=38, 
i=39, 
i=40, 
i=41, 
i=42, 
i=43, 
i=44, 
i=45, 
i=46, 
i=47, 
i=48, 
i=49, 
i=50, 
i=51, 
i=52, 
i=53, 
i=54, 
i=55, 
i=56, 
i=57, 
i=58, 
i=59, 
i=60, 
i=61, 
i=62, 
i=63, 
i=64, 
i=65, 
i=66, 
i=67, 
i=68, 
i=69, 
i=70, 
i=71, 
i=72, 
i=73, 
i=74, 
i=75, 
i=76, 
i=77, 
i=78, 
i=79, 
i=80, 
i=81, 
i=82, 
i=83, 
i=84, 
i=85, 
i=86, 
i=87, 
i=88, 
i=89, 
i=90, 
i=91, 
i=92, 
i=93, 
i=94, 
i=95, 
i=96, 
i=97, 
i=98, 
i=99, 
i=100, 
i=101, 
i=102, 
i=103, 
i=104, 
i=105, 
i=106, 
i=107, 
i=108, 
i=109, 
i=110, 
i=111, 
i=112, 
i=113, 
i=114, 
i=115, 
i=116, 
i=117, 
i=118, 
i=119, 


# install additional packages

In [ ]:
# !cd ..
# !source .venv/bin/activate
# !poetry add readability-lxml
# !poetry install

In [31]:
# !cd ..
# !source .venv/bin/activate
# !poetry add tabulate
# !poetry install

/bin/bash: line 1: .venv/bin/activate: No such file or directory
Using version ^0.9.0 for tabulate

Updating dependencies
Resolving dependencies... (0.8s)

Package operations: 1 install, 0 updates, 0 removals

  - Installing tabulate (0.9.0): Pending...
  - Installing tabulate (0.9.0): Downloading... 0%
  - Installing tabulate (0.9.0): Downloading... 100%
  - Installing tabulate (0.9.0): Installing...
  - Installing tabulate (0.9.0)

Writing lock file
Installing dependencies from lock file

No dependencies to install or update

Installing the current project: finance-rag-assistant (0.1.0)Installing the current project: finance-rag-assistant (0.1.0)
